In [1]:
!pip install qsimov

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 980.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.8/170.8 kB 3.9 MB/s eta 0:00:00ta 0:00:01


In [2]:
import numpy as np
import qsimov as qs
import time
import unittest

In [3]:
def to_bin(a,n):
    base = np.full(n,2)
    exponent = np.arange(0,n,1)
    result = base ** exponent
    temp = a // result
    positions = temp & 1
    return positions

In [4]:
def get_rs():
    gate = qs.QGate(3, 0, "RS")
    gate.add_operation("X", targets=2, controls=0) 
    gate.add_operation("X", targets=2, controls=1) 
    return gate

In [5]:
def get_ra(n):
    num_qubits = 3*n+1
    gate = qs.QGate(num_qubits, 0, "RA") # Reverse Adder
    carry = get_carry()
    rs = get_rs()
    for i in range(n-1):
        temp = 3*i
        gate.add_operation(rs,targets=[temp,temp+1,temp+2])
        gate.add_operation(carry, targets=[temp,temp+1,temp+2,temp+3])
    temp += 3
    gate.add_operation(rs,targets=[temp,temp+1,temp+2])
    gate.add_operation("X", targets=temp+2, controls=temp+1)
    rc = get_rc() # get reverse carry
    gate.add_operation(rc,targets=[temp,temp+1,temp+2,temp+3])
    for i in range(n-2,-1,-1):
        temp = 3*i
        gate.add_operation(rc,[temp,temp+1,temp+2,temp+3])
    return gate

In [6]:
# Page 166 Stinson DR. Cryptography: Theory and Practice, Third Edition. Chapman & Hall/CRC; 2006.
def ext_eucl_alg(a,b):
    a0 = a
    b0 = b
    t0 = 0
    t = 1
    s0 = 1
    s = 0
    q = np.floor(a0/b0)
    r = a0 - q*b0
    while r>0:
        temp = t0 - q*t
        t0 = t
        t = temp
        temp = s0 - q*s
        s0 = s
        s = temp
        a0 = b0
        b0 = r
        q = np.floor(a0/b0)
        r = a0 - q*b0
    r = b0
    return r,s,t
    # r = gcd(a,b) and sa+tb = r

In [7]:
def mod_inv(b,n):
    r,s,t = ext_eucl_alg(b,n)
    if r == 1:
        if not(0<=s<n):
            s = s % n 
        s = s.astype(int)
        return s
    return None

In [8]:
def get_sum():
    gate = qs.QGate(3, 0, "SUM")
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=2, controls=0)
    return gate

In [9]:
def get_carry():
    gate = qs.QGate(4, 0, "CARRY")
    gate.add_operation("X", targets=3, controls=[1,2])
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=3, controls=[0,2])
    return gate

In [10]:
def get_rc():
    gate = qs.QGate(4, 0, "RC")
    gate.add_operation("X", targets=3, controls=[0,2])
    gate.add_operation("X", targets=2, controls=1)
    gate.add_operation("X", targets=3, controls=[1,2])
    return gate

In [11]:
def get_adder(n):
    num_qubits = 3*n+1
    gate = qs.QGate(num_qubits, 0, "ADDER")
    carry = get_carry()
    for i in range(n-1):
        temp = 3*i
        gate.add_operation(carry, targets=[temp,temp+1,temp+2,temp+3])
    temp += 3
    gate.add_operation(carry,targets=[temp,temp+1,temp+2,temp+3])
    gate.add_operation("X", targets=temp+2, controls=temp+1)
    sum = get_sum()
    gate.add_operation(sum,targets=[temp,temp+1,temp+2])
    rc = get_rc() # get reverse carry
    for i in range(n-2,-1,-1):
        temp = 3*i
        gate.add_operation(rc,[temp,temp+1,temp+2,temp+3])
        gate.add_operation(sum, targets=[temp,temp+1,temp+2])
    return gate

In [12]:
def get_modadd(N,n):
    num_qubits = 4*n+2
    gate = qs.QGate(num_qubits, 0, "MODADDER") # Modular Adder
    
    # first adder
    adder = get_adder(n)
    num_qubits_adder = 3*n+1
    targets = np.arange(0,num_qubits_adder,1)
    gate.add_operation(adder,targets=targets)
    
    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])

    # reverse adder
    ra = get_ra(n)
    gate.add_operation(ra,targets=targets)

    #anticontrol
    target = num_qubits-1
    anticontrol = num_qubits_adder-1
    gate.add_operation("X", targets=target, anticontrols=anticontrol)

    #cancel N
    control = target
    exponent = np.arange(0,n,1)
    indices = exponent*3+1
    positions = to_bin(N,n)
    positions = positions == 1
    targets = indices[positions]
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    # second adder
    targets_ = np.arange(0,num_qubits_adder,1)
    gate.add_operation(adder,targets=targets_)


    #reassign N
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)



    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])



    # reverse adder
    gate.add_operation(ra,targets=targets_)


    #control
    target = num_qubits-1
    control = num_qubits_adder-1
    gate.add_operation("X", targets=target, controls=control)

    
    #third adder
    gate.add_operation(adder,targets=targets_)

    return gate

In [13]:
def get_mod_mult(a,N,n):
    num_qubits = 5*n+3
    gate = qs.QGate(num_qubits, 0, "MODMULT") # Modular Multiplier
    num_qubits_mod_add = 4*n+2
    targets_mod_add = np.arange(0,num_qubits_mod_add,1) + n + 1
    exponent = np.arange(0,n,1)
    result = 2 ** exponent
    temp = a * result
    temp = temp % N
    indices = np.arange(0,n,1)
    indices = indices*3+1
    indices = indices + n + 1
    for i in range(n):
        temp2 = temp[i] // result
        positions = temp2 & 1
        positions = positions == 1
        targets = indices[positions]
        #print(targets)
        for j in targets:
            gate.add_operation("X", targets=j, controls=[0,i+1])
        modadd = get_modadd(N,n)
        gate.add_operation(modadd, targets=targets_mod_add)
        for j in targets:
            gate.add_operation("X", targets=j, controls=[0,i+1])
    for i in range(n):
        gate.add_operation("X", targets=i*3+2+n+1, anticontrols=0,controls=i+1)
    return gate

In [14]:
def get_rev_modadd(N,n):
    num_qubits = 4*n+2
    gate = qs.QGate(num_qubits, 0, "RMA") # Reverse Modular Adder

    # reverse adder
    ra = get_ra(n)
    num_qubits_adder = 3*n+1
    targets = np.arange(0,num_qubits_adder,1)
    gate.add_operation(ra,targets=targets)

    # control
    target = num_qubits-1
    control = num_qubits_adder-1
    gate.add_operation("X", targets=target, controls=control)


    # adder
    adder = get_adder(n)
    gate.add_operation(adder,targets=targets)


    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n-1,-1,-1):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])


    #reassign N
    control = target
    exponent = np.arange(0,n,1)
    indices = exponent*3+1
    positions = to_bin(N,n)
    positions = positions == 1
    targets = indices[positions]
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    # reverse adder
    targets_ = np.arange(0,num_qubits_adder,1)
    gate.add_operation(ra,targets=targets_)


    #cancel N
    for i in targets:
        gate.add_operation("X", targets=i, controls=control)


    #anticontrol
    anticontrol = num_qubits_adder-1
    target = num_qubits-1
    gate.add_operation("X", targets=target, anticontrols=anticontrol)


    # adder
    gate.add_operation(adder,targets=targets_)


    #swaps
    positions = np.arange(0,n,1) + num_qubits_adder
    for i in range(n-1,-1,-1):
        from_ = i*3+1
        to = positions[i]
        gate.add_operation("SWAP", targets=[from_,to])
    
    # reverse adder
    gate.add_operation(ra,targets=targets_)

    return gate

In [15]:
def get_rev_mod_mult(a,N,n):
    num_qubits = 5*n+3
    gate = qs.QGate(num_qubits, 0, "RMM") # Reverse Modular Multiplier

    for i in range(n-1,-1,-1):
        gate.add_operation("X", targets=i*3+2+n+1, anticontrols=0,controls=i+1)
    num_qubits_mod_add = 4*n+2
    targets_mod_add = np.arange(0,num_qubits_mod_add,1) + n + 1
    exponent = np.arange(0,n,1)
    result = 2 ** exponent
    temp = a * result
    temp = temp % N
    indices = np.arange(0,n,1)
    indices = indices*3+1
    indices = indices + n + 1
    for i in range(n-1,-1,-1):
        temp2 = temp[i] // result
        positions = temp2 & 1
        positions = positions == 1
        targets = indices[positions]
        #print(targets)
        for j in targets:
            gate.add_operation("X", targets=j, controls=[0,i+1])
        rev_modadd = get_rev_modadd(N,n)
        gate.add_operation(rev_modadd, targets=targets_mod_add)
        for j in targets:
            gate.add_operation("X", targets=j, controls=[0,i+1])
    return gate

In [16]:
def get_mod_exp(a,N,n):
    num_qubits = 6*n+3
    gate = qs.QGate(num_qubits, 0, "EXPMOD") # Modular Exponentiation
    gate.add_operation("X", targets=n+1) # Initialize Register 1 to 1

    num_qubits_mod_mult = 5*n+3

    for i in range(n):
        #control qubit
        gate.add_operation("X", targets=n, controls=i)
        
        # MODMULT
        temp = a ** (2 ** i)
        temp = temp % N
        
        targets = np.arange(0,num_qubits_mod_mult,1)
        targets += n 
        modmult = get_mod_mult(temp,N,n)
        gate.add_operation(modmult, targets=targets)

        # SWAP
        register_one_indices = np.arange(0,n,1)
        register_one_indices += n
        register_one_indices += 1
        
        register_b_indices = np.arange(0,n,1)
        register_b_indices *=3
        register_b_indices += 2
        register_b_indices += 2*n
        register_b_indices += 1 

        for j in range(n):
            from_ = register_one_indices[j]
            to = register_b_indices[j]
            gate.add_operation("SWAP", targets=[from_,to])


        # Reverse MODMULT
        temp = a ** (2 ** i)
        temp = temp % N
        temp = mod_inv(temp,N)
        rev_mod_mult = get_rev_mod_mult(temp,N,n)
        gate.add_operation(rev_mod_mult, targets=targets)

        # control qubit
        gate.add_operation("X", targets=n, controls=i)
    return gate

In [17]:
def expomod(a, n, z):
  i = n
  r = 1
  x = a % z
  while i > 0:
    if i % 2 == 1:
      r = (r*x) % z
    x = (x ** 2) % z
    i = i // 2
  return r

In [19]:
def test_mod_exp_four_qubits(a,x,N,n):
                  size = 6*n+3
                  register_2_size = n
                  register_1_plus_control = n + 1
                  adder_bits = 3*n+1
    
                  # modulo
                  modulo_bin= to_bin(N,n)
                  indices_modulo = np.arange(0,n,1)
                  indices_modulo += register_2_size
                  indices_modulo += register_1_plus_control
                  indices_modulo += adder_bits
                  expected = expomod(a,x,N)
        
        
                  input = np.zeros(size)
                  input[indices_modulo] = modulo_bin
        
                  # exponent
                  exponent = x
                  indices_exponent = np.arange(0,n,1)
                  exponent_bin= to_bin(exponent,n)
                  input[indices_exponent] = exponent_bin
        
                  c = qs.QCircuit(size, size, "EXPMOD", ancilla=input)
                  mod_exp = get_mod_exp(a,N,n)
                  targets = np.arange(0,size,1)
                  c.add_operation(mod_exp,targets=targets)
                  c.add_operation("MEASURE", targets=targets, outputs=targets)
                  use_system = False
                  executor = qs.Drewom(qmachine="doki",
                                      extra={"num_threads": -1,
                                              "random_generator": np.random.rand,
                                              "use_system": use_system})
                  circuit = c
                  iterations = 1
                  t = time.time()
                  mes = executor.execute(circuit, iterations=iterations)
                  t = time.time() - t
                  hours = t // 3600
                  t %= 3600
                  minutes = t // 60
                  t %= 60
                  seconds = t

                  indices = np.arange(0,n,1)
                  indices += register_2_size
                  indices += 1
                  mes = np.array(mes[0])
        
        
                  temp = mes[indices]
                  weights = np.arange(0,n,1)
                  weights = 2 ** weights
                  temp = temp * weights
                  actual = np.sum(temp)
        
                  indices_register_a = np.arange(0,n,1)
                  indices_register_a *= 3
                  indices_register_a += 1
                  indices_register_a += register_2_size
                  indices_register_a += register_1_plus_control
        
                  temp = mes[indices_register_a]
                  temp = temp * weights
                  register_a = np.sum(temp)
        
                  temp = mes[indices_modulo]
                  temp = temp * weights
                  modulo = np.sum(temp)
        
                  temp = mes[indices_exponent]
                  temp = temp * weights
                  register_2 = np.sum(temp)
        
                  # assert
                  #print(i,j)
                  #self.assertEqual(register_2,x)
                  print(register_2==x)
                  #self.assertEqual(mes[register_2_size],0) # control qubit
                  print(mes[register_2_size]==0)
                  #self.assertEqual(actual,expected)
                  print("Actual:",actual)
                  print("Expected:",expected)
                  print(actual==expected)
                  #self.assertEqual(register_a,0)
                  print(register_a==0)
                  #self.assertEqual(modulo,N)
                  print(modulo==N)
                  #self.assertEqual(mes[size-1],0) # temp qubit
                  print(mes[size-1]==0)
                  print(f'After {hours} hours, {minutes} minutes and {seconds} seconds, {a}^{x} mod {N} = {actual}')

In [23]:
test_mod_exp_four_qubits(2,1,15,4)

True
True
Actual: 2
Expected: 2
True
True
True
True
After 2.0 hours, 43.0 minutes and 33.66354155540466 seconds, 2^1 mod 15 = 2


In [20]:
test_mod_exp_four_qubits(2,2,15,4)

True
True
Actual: 4
Expected: 4
True
True
True
True
After 2.0 hours, 38.0 minutes and 7.400023698806763 seconds, 2^2 mod 15 = 4


In [21]:
test_mod_exp_four_qubits(2,3,15,4)

True
True
Actual: 8
Expected: 8
True
True
True
True
After 2.0 hours, 38.0 minutes and 28.900103330612183 seconds, 2^3 mod 15 = 8


In [22]:
test_mod_exp_four_qubits(2,4,15,4)

True
True
Actual: 1
Expected: 1
True
True
True
True
After 2.0 hours, 37.0 minutes and 49.79809379577637 seconds, 2^4 mod 15 = 1


In [23]:
test_mod_exp_four_qubits(7,4,15,4)

True
True
Actual: 1
Expected: 1
True
True
True
True
After 2.0 hours, 37.0 minutes and 2.018026113510132 seconds, 7^4 mod 15 = 1


In [ ]:
class TestModularExponentiation(unittest.TestCase):
    def test_mod_exp_three_qubits(self):
            N = 21
            n = 5
    
            size = 6*n+3
            register_2_size = n
            register_1_plus_control = n + 1
            adder_bits = 3*n+1
    
            # modulo
            modulo_bin= to_bin(N,n)
            indices_modulo = np.arange(0,n,1)
            indices_modulo += register_2_size
            indices_modulo += register_1_plus_control
            indices_modulo += adder_bits
            for a in [2,5]:
                for x in range(2**n):
        
                  expected = expomod(a,x,N)
        
        
                  input = np.zeros(size)
                  input[indices_modulo] = modulo_bin
        
                  # exponent
                  exponent = x
                  indices_exponent = np.arange(0,n,1)
                  exponent_bin= to_bin(exponent,n)
                  input[indices_exponent] = exponent_bin
        
                  c = qs.QCircuit(size, size, "EXPMOD", ancilla=input)
                  mod_exp = get_mod_exp(a,N,n)
                  targets = np.arange(0,size,1)
                  c.add_operation(mod_exp,targets=targets)
                  c.add_operation("MEASURE", targets=targets, outputs=targets)
                  use_system = False
                  executor = qs.Drewom(qmachine="doki",
                                      extra={"num_threads": -1,
                                              "random_generator": np.random.rand,
                                              "use_system": use_system})
                  circuit = c
                  iterations = 1
                  mes = executor.execute(circuit, iterations=iterations)
        
                  indices = np.arange(0,n,1)
                  indices += register_2_size
                  indices += 1
                  mes = np.array(mes[0])
        
        
                  temp = mes[indices]
                  weights = np.arange(0,n,1)
                  weights = 2 ** weights
                  temp = temp * weights
                  actual = np.sum(temp)
        
                  indices_register_a = np.arange(0,n,1)
                  indices_register_a *= 3
                  indices_register_a += 1
                  indices_register_a += register_2_size
                  indices_register_a += register_1_plus_control
        
                  temp = mes[indices_register_a]
                  temp = temp * weights
                  register_a = np.sum(temp)
        
                  temp = mes[indices_modulo]
                  temp = temp * weights
                  modulo = np.sum(temp)
        
                  temp = mes[indices_exponent]
                  temp = temp * weights
                  register_2 = np.sum(temp)
        
                  # assert
                  #print(i,j)
                  self.assertEqual(register_2,x)
                  self.assertEqual(mes[register_2_size],0) # control qubit
                  self.assertEqual(actual,expected)
                  self.assertEqual(register_a,0)
                  self.assertEqual(modulo,N)
                  self.assertEqual(mes[size-1],0) # temp qubit